# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hanizakkk/flyrank_working-repo/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**June 2026 live scoring queue (208,636 items, no label yet - forward-looking):**

| Action | Count | Meaning |
|---|---|---|
| `refresh_priority` | 59,351 | Model score >= 0.66 - visible, underperforming, worth review first |
| `monitor` | 86,035 | Model score 0.40-0.66 - watch, not urgent |
| `protect` | 2,221 | Model score < 0.40 among visible items - currently doing fine, leave alone |
| `insufficient_data_manual_review` | 61,029 | Below the 20-impression noise floor - too sparse to score confidently |

Reason codes carried from the same rule as ML-07: `weak_position`, `low_ctr`
(can combine), `general_review`, `low_visibility_monitor_only`.

In [1]:
import pandas as pd
ranked_queue = pd.read_json("../outputs/ranked_recommendations_sample.json")
print(ranked_queue["action"].value_counts())
ranked_queue.head(20)

action
refresh_priority    2000
Name: count, dtype: int64


,rank,client_hash_id,content_hash_id,model_score,action,reason_code,confidence,impressions,clicks,avg_position,ctr
0,1,client_b10cb2997d0c7c86,content_87eb4619444da89f,0.833023,refresh_priority,weak_position|low_ctr,high,646,0,28.099554,0.000000
1,2,client_810019792c9b8efc,content_826a6d837682df3c,0.833020,refresh_priority,weak_position|low_ctr,high,498,0,10.845758,0.000000
2,3,client_9958f0a7ae1df715,content_572263a160b6c8ab,0.830888,refresh_priority,weak_position|low_ctr,high,510,0,26.276404,0.000000
3,4,client_73cda7b4e4f265ea,content_1f10cce14fa14a4c,0.830485,refresh_priority,low_ctr,high,338,0,9.095340,0.000000
4,5,client_9958f0a7ae1df715,content_d8288f22519f7d53,0.830259,refresh_priority,weak_position|low_ctr,high,633,0,38.063023,0.000000
5,6,client_b10cb2997d0c7c86,content_5d2c0a2568ca0bac,0.828595,refresh_priority,low_ctr,high,1315,0,8.921163,0.000000
6,7,client_73cda7b4e4f265ea,content_ebe1f0da038a3a54,0.826721,refresh_priority,weak_position|low_ctr,high,2283,2,15.898636,0.000876
7,8,client_b10cb2997d0c7c86,content_8b1d28df3ed4e324,0.825458,refresh_priority,weak_position|low_ctr,high,1176,1,38.878251,0.000850
8,9,client_23a62021009f63c4,content_11a15d9c9ff6f9ea,0.824141,refresh_priority,weak_position|low_ctr,high,651,0,30.984887,0.000000
9,10,client_fef1a8f436438636,content_3aba7dc3b43d583f,0.823872,refresh_priority,weak_position|low_ctr,high,926,0,21.630945,0.000000


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this:** a content/SEO lead deciding where to spend limited
refresh-review time this month - not an auto-publish pipeline.

**Where it stops being valid:** this queue describes *observed, decision-time*
search signals only. It says nothing about *why* a page underperforms
(content quality, a SERP feature change, a competitor, seasonality), doesn't
know about anything outside GSC/GA4 (no editorial calendar, no product
changes), and the 0.66/0.40 action thresholds are fixed cutoffs chosen for
readability, not tuned to any specific team's review capacity - a team with
less bandwidth should raise the `refresh_priority` bar, not treat 59,351
items as a literal to-do list.

In [2]:
print("Documented in the markdown cell above - no numeric check needed here.")

Documented in the markdown cell above - no numeric check needed here.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**No-go list - never auto-act on:**
- Anything under the 20-impression noise floor (`insufficient_data_manual_review`,
  61,029 items) - the signal-to-noise ratio there is too poor to trust a score.
- Anything the model flags `refresh_priority` with `avg_position` between 9
  and 11 - this is exactly the boundary region ML-08 found the most
  confident-but-wrong predictions in (see w05, section 4).
- Any single client's top-ranked items in isolation, without checking whether
  that client's whole portfolio is skewed (a client having a rough month
  everywhere isn't the same signal as one specific page underperforming).

**Human review before acting:** a person should sanity-check the top 20-30
items in any given `refresh_priority` batch against actual page content
before committing review time - the model has never seen the page itself,
only its numbers.

In [3]:
noise_floor_count = (ranked_queue["action"] == "insufficient_data_manual_review").sum()
boundary_flags = ranked_queue[(ranked_queue["action"] == "refresh_priority") & (ranked_queue["avg_position"].between(9, 11))]
print(f"Routed to manual review (below noise floor): {noise_floor_count}")
print(f"refresh_priority items sitting in the unstable avg_position 9-11 boundary: {len(boundary_flags)}")

Routed to manual review (below noise floor): 0
refresh_priority items sitting in the unstable avg_position 9-11 boundary: 158


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Retrain trigger:** re-run this pipeline monthly as each new decision/outcome
month pair becomes available (e.g. April->May once May closes out), and
compare precision@50 against the March->April dev number (0.76 baseline,
0.86 model) and the May->June sealed-test number (0.86). If precision@50 on a
fresh month drops meaningfully below those, or the action-category mix shifts
sharply month over month without an obvious cause (a warehouse schema change,
a major algorithm update, a big shift in which clients have data), treat that
as a signal the scoring rule needs review before the next batch goes out.

In [4]:
import json
with open("../outputs/action_playbook_summary.json") as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))
print("\nReference dev/sealed numbers to compare future runs against:")
with open("../outputs/model_metrics.json") as f:
    print(json.load(f))
with open("../outputs/sealed_test_result.json") as f:
    print(json.load(f))

{
  "action_counts": {
    "insufficient_data_manual_review": 61029,
    "monitor": 86035,
    "protect": 2221,
    "refresh_priority": 59351
  },
  "decision_month": "2026-06",
  "n_scored": 208636,
  "noise_floor_impressions": 20
}

Reference dev/sealed numbers to compare future runs against:
[{'base_rate': 0.6620141722138716, 'model': 'baseline_rule', 'precision_at_50': 0.7}, {'base_rate': 0.6620141722138716, 'model': 'logistic_regression', 'precision_at_50': 0.84}, {'base_rate': 0.6620141722138716, 'model': 'random_forest', 'precision_at_50': 0.86}]
{'base_rate': 0.7282517565732892, 'decision_month': '2026-05', 'n_rows': 184877, 'outcome_month': '2026-06', 'precision_at_50': 0.86}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Already exported by the pipeline run: `work/outputs/ranked_recommendations_sample.json`,
`work/outputs/action_playbook_summary.json`,
`work/figures/action_mix.svg`, `work/figures/feature_importance.svg`.

In [5]:
import os
for f in ["../outputs/ranked_recommendations_sample.json", "../outputs/action_playbook_summary.json",
          "../figures/action_mix.svg", "../figures/feature_importance.svg"]:
    print(f, "exists:", os.path.exists(f))

../outputs/ranked_recommendations_sample.json exists: True
../outputs/action_playbook_summary.json exists: True
../figures/action_mix.svg exists: True
../figures/feature_importance.svg exists: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.